# The Datetime Accessor (`.dt`) & Resampling

Just like string columns have a dedicated `.str` accessor to perform custom text operations, datetime columns in Pandas have a powerful **`.dt` accessor**.

Once a column is converted to a datetime type, the `.dt` property allows you to extract specific date components (like the year, month, day, or name of the weekday) on the fly without writing custom parsing functions.

Additionally, we will learn how to **resample** temporal data. If you have daily transactions but want to see aggregated monthly totals, you can use `.resample()`, which acts like a specialized `.groupby()` tailored for temporal frequencies.

### Simple Explanation & Real-World Analogy
Think of a true datetime object as a smart **Swiss Army Knife**. If you hand it a date, you don't just get a label; you can pull out different specialized blades: one to tell you the exact calendar year, another to tell you if it was a Leap Year, and another to look up if that date fell on a Tuesday. The `.dt` accessor is how you deploy those specialized blades with a single click.

### Code Examples

Let's load a dataset of daily transaction entries and extract custom features using the `.dt` accessor.


In [6]:
import pandas as pd

# Creating a datetime index DataFrame
sales_data = {
    'Date_Str': ['2026-08-23', '2026-08-24', '2026-12-25', '2026-12-31'],
    'Units_Sold': [10, 15, 50, 80]
}
df = pd.DataFrame(sales_data)
df['Date'] = pd.to_datetime(df['Date_Str'])

# Extracting components using the .dt accessor
df['Year'] = df['Date'].dt.year
df['Month_Name'] = df['Date'].dt.month_name()
df['Day_Name'] = df['Date'].dt.day_name()
df['Is_Leap_Year'] = df['Date'].dt.is_leap_year

print(df[['Date', 'Year', 'Month_Name', 'Day_Name', 'Is_Leap_Year']])

        Date  Year Month_Name  Day_Name  Is_Leap_Year
0 2026-08-23  2026     August    Sunday         False
1 2026-08-24  2026     August    Monday         False
2 2026-12-25  2026   December    Friday         False
3 2026-12-31  2026   December  Thursday         False


#### Aggregating and Resampling Dates (`.resample`)
To aggregate dates by specific frequencies (e.g., Weekly, Monthly, Quarterly), we first set the datetime column as the DataFrame's index. Once the index is temporal, we can apply **`.resample()`**.

In [9]:
# set_index() returns a NEW DataFrame; we do NOT use inplace=True.
# That keeps 'df' untouched, so this cell can be re-run any number of times.
df_by_date = df.set_index('Date')

# Creating more daily records for a clearer resample representation
daily_sales = pd.DataFrame({
    'Revenue': [100, 200, 150, 300, 250, 400, 350]
}, index=pd.to_datetime([
    '2026-08-01', '2026-08-02', '2026-08-03',
    '2026-09-01', '2026-09-02',
    '2026-10-01', '2026-10-02'
]))

# Resample monthly ('ME') and calculate the sum of revenues
monthly_summary = daily_sales.resample('ME').sum()  # 'ME' stands for Month End frequency
print("--- Monthly Resampled Revenue ---")
print(monthly_summary)

--- Monthly Resampled Revenue ---
            Revenue
2026-08-31      450
2026-09-30      550
2026-10-31      750


### Common Pitfalls to Avoid

1. **Calling `.dt` on a Non-Datetime Column**: If you try to run `df['Date_Str'].dt.year` on a raw string column, Pandas will raise an `AttributeError`. You must convert the column with `pd.to_datetime()` first.
2. **Forgetting to Set Datetime Index Before Resampling**: `.resample()` *only* works if the index of your DataFrame is a true DatetimeIndex . If you try to resample with a default integer index, Pandas will raise a `TypeError`.

#### Exercise 1 (Easy)
Extract the name of the day of the week (e.g., `'Monday'`, `'Tuesday'`) from the following Series of dates.
```python
dates_series = pd.Series(pd.to_datetime(['2026-01-01', '2026-01-02', '2026-01-03']))
```

In [10]:
import pandas as pd
dates_series = pd.Series(pd.to_datetime(['2026-01-01', '2026-01-02', '2026-01-03']))

# Using .dt.day_name() to extract weekday names
weekday_names = dates_series.dt.day_name()
print(weekday_names)

0    Thursday
1      Friday
2    Saturday
dtype: str


#### Exercise 2 (Medium)
Given the following daily dataset, set the date as an index, and resample the data to find the **weekly mean** of the recorded temperatures.
```python
weather_df = pd.DataFrame({
    'Date_Str': ['2026-08-01', '2026-08-02', '2026-08-08', '2026-08-09'],
    'Temperature': [32, 34, 28, 30]
})
```

In [11]:
import pandas as pd
weather_df = pd.DataFrame({
    'Date_Str': ['2026-08-01', '2026-08-02', '2026-08-08', '2026-08-09'],
    'Temperature': [32, 34, 28, 30]
})

# 1. Parse string column to datetime
weather_df['Date'] = pd.to_datetime(weather_df['Date_Str'])

# 2. Set index as the datetime column [227, 582]
weather_df.set_index('Date', inplace=True)

# 3. Resample weekly ('W' or 'W-SUN') and calculate mean
weekly_temp = weather_df['Temperature'].resample('W').mean()
print(weekly_temp)

Date
2026-08-02    33.0
2026-08-09    29.0
Freq: W-SUN, Name: Temperature, dtype: float64
